# R Master v9 · Lap vs Mona Body Compare

**纯诊断版，不接头、不改源模型、不烘 Rest Pose、不转权重、不导 VRM。**

目标：把 Lapine BaseBody 与当前 Mona true body 做统一高度/中心后的轮廓叠加与关键比例对比，判断 Lap 是否值得作为 R 的完整外观/体型 donor；如果不合适，再回到 Lap 头 + Mona 身体路线。

输出：正/侧/背/3⁄4，全身 + 胸腰、腰骨盆大腿根、头颈切线，共 7 张图；另附 JSON 数据报告和日志。


In [ ]:

from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, sys, json, shutil, zipfile, hashlib, subprocess, textwrap, urllib.request, tarfile

ROOT=Path("/content/drive/MyDrive/R_Master")
OUT=ROOT/"v9_body_compare"/"latest"
OUT.mkdir(parents=True,exist_ok=True)
LOCAL=Path("/content/r_master_v9"); LOCAL.mkdir(parents=True,exist_ok=True)
CACHE=ROOT/"cache"; CACHE.mkdir(parents=True,exist_ok=True)

# Mona: prefer v7c body, then v8c preview, then v2.
candidates=[
 ROOT/"v7c_refine/latest/R_Master_v7c_REFINED_PREVIEW.blend",
 ROOT/"v8c_head_beauty/latest/R_Master_v8c_HEAD_BEAUTY_PREVIEW.blend",
 ROOT/"v2/latest/R_Master_Align_v2_PREVIEW.blend",
]
MONA=next((p for p in candidates if p.exists()),None)
if MONA is None: raise FileNotFoundError("Mona preview source not found in Drive")

# Lapine zip cached from prior steps.
lap_candidates=[
 ROOT/"reference/Lapine_Ver.1.11_A.zip",
 ROOT/"cache/Lapine_Ver.1.11_A.zip",
 ROOT/"_cache/Lapine_Ver.1.11_A.zip",
 ROOT/"Lapine_Ver.1.11_A.zip",
 Path("/content/drive/MyDrive/Lapine_Ver.1.11_A.zip"),
]
LAPZIP=next((p for p in lap_candidates if p.exists()),None)
if LAPZIP is None:
    raise FileNotFoundError("Lapine_Ver.1.11_A.zip not found. Reuse the same Drive cache used by v8.")

# Blender cache/install.
BLENDER_VERSION="4.4.3"
BDIR=CACHE/f"blender-{BLENDER_VERSION}-linux-x64"
BLENDER=BDIR/"blender"
if not BLENDER.exists():
    arc=LOCAL/f"blender-{BLENDER_VERSION}.tar.xz"
    url=f"https://download.blender.org/release/Blender4.4/blender-{BLENDER_VERSION}-linux-x64.tar.xz"
    urllib.request.urlretrieve(url,arc)
    with tarfile.open(arc,"r:xz") as t: t.extractall(CACHE)
if not BLENDER.exists(): raise FileNotFoundError(BLENDER)

# Extract Lapine FBX directly from unitypackage.
EX=LOCAL/"lap"; EX.mkdir(exist_ok=True)
UPKG=EX/"Lapine.unitypackage"
FBX=EX/"Lapine_BaseBody.fbx"
if not FBX.exists():
    with zipfile.ZipFile(LAPZIP) as z:
        member=next(n for n in z.namelist() if n.endswith("Lapine/Lapine.unitypackage"))
        UPKG.write_bytes(z.read(member))
    with tarfile.open(UPKG,"r:gz") as t:
        members=t.getmembers()
        path_by_dir={}
        asset_by_dir={}
        for m in members:
            parts=m.name.split("/")
            if len(parts)!=2: continue
            d,f=parts
            if f=="pathname":
                path_by_dir[d]=t.extractfile(m).read().decode("utf-8","ignore").strip()
            elif f=="asset":
                asset_by_dir[d]=m
        target=None
        for d,p in path_by_dir.items():
            if p.endswith("Assets/Models/FBX/Lapine_BaseBody.fbx") or p.endswith("Lapine_BaseBody.fbx"):
                target=d; break
        if target is None: raise RuntimeError("Lapine_BaseBody.fbx not found inside unitypackage")
        FBX.write_bytes(t.extractfile(asset_by_dir[target]).read())

SCRIPT=LOCAL/"compare.py"
SCRIPT.write_text(r'''
import bpy,sys,os,json,math
from mathutils import Vector
argv=sys.argv[sys.argv.index("--")+1:]
def arg(k):
    i=argv.index(k); return argv[i+1]
mona=arg("--mona"); lap=arg("--lap"); out=arg("--out")
os.makedirs(out,exist_ok=True)

# Open Mona source and isolate true body.
bpy.ops.wm.open_mainfile(filepath=mona)
body=bpy.data.objects.get("R2_Mona_Main") or bpy.data.objects.get("Mona_Main")
rig=bpy.data.objects.get("R_Master_Align_v2_PREVIEW") or bpy.data.objects.get("Mona_Armature")
if not body or not rig: raise RuntimeError("Mona body/rig not found")

# Delete every mesh except Mona body for clean diagnostic.
for o in list(bpy.data.objects):
    if o.type=="MESH" and o!=body: bpy.data.objects.remove(o,do_unlink=True)
body.name="CMP_Mona"
for p in body.data.polygons:p.use_smooth=True

# Import Lap BaseBody.
bpy.ops.import_scene.fbx(filepath=lap)
lap_meshes=[o for o in bpy.context.scene.objects if o.type=="MESH" and o!=body]
if not lap_meshes: raise RuntimeError("Lap BaseBody import produced no mesh")
# Choose largest mesh by vertices as body shell.
lapbody=max(lap_meshes,key=lambda o:len(o.data.vertices))
lapbody.name="CMP_Lap"
for o in list(lap_meshes):
    if o!=lapbody: bpy.data.objects.remove(o,do_unlink=True)
for p in lapbody.data.polygons:p.use_smooth=True

# World bounds helper.
def pts(o):
    return [o.matrix_world@v.co for v in o.data.vertices]
def bounds(o):
    p=pts(o); mn=Vector((min(x.x for x in p),min(x.y for x in p),min(x.z for x in p))); mx=Vector((max(x.x for x in p),max(x.y for x in p),max(x.z for x in p))); return mn,mx
mmn,mmx=bounds(body); lmn,lmx=bounds(lapbody)
mh=mmx.z-mmn.z; lh=lmx.z-lmn.z
if mh<=0 or lh<=0: raise RuntimeError("bad bounds")

# Normalize Lap to Mona body height and floor/center, for silhouette comparison only.
s=mh/lh
lapbody.scale*=s
bpy.context.view_layer.objects.active=lapbody
bpy.ops.object.transform_apply(location=False,rotation=False,scale=True)
lmn,lmx=bounds(lapbody)
mcenter=(mmn+mmx)*.5; lcenter=(lmn+lmx)*.5
lapbody.location += Vector((mcenter.x-lcenter.x,mcenter.y-lcenter.y,mmn.z-lmn.z))
bpy.context.view_layer.update()
lmn,lmx=bounds(lapbody)

# Materials: Mona gray, Lap translucent cyan-like neutral via alpha.
def mat(name,color,alpha=1):
    m=bpy.data.materials.new(name);m.diffuse_color=(*color,alpha);m.use_nodes=True
    bs=m.node_tree.nodes.get("Principled BSDF");bs.inputs["Base Color"].default_value=(*color,1);bs.inputs["Roughness"].default_value=.8
    if alpha<1:
        bs.inputs["Alpha"].default_value=alpha
        try:m.surface_render_method="DITHERED"
        except:pass
    return m
body.data.materials.clear();body.data.materials.append(mat("Mona",(0.42,.42,.42),1))
lapbody.data.materials.clear();lapbody.data.materials.append(mat("Lap",(0.55,.70,.72),.48))

# Measurements by horizontal slabs: width/depth at normalized heights.
def slab(o,zfrac,band=.012):
    mn,mx=bounds(o); z=mn.z+(mx.z-mn.z)*zfrac; tol=(mx.z-mn.z)*band
    q=[p for p in pts(o) if abs(p.z-z)<=tol]
    if not q:return None
    return {"width":max(p.x for p in q)-min(p.x for p in q),"depth":max(p.y for p in q)-min(p.y for p in q),"z":z}
levels={"shoulder":.80,"chest":.70,"waist":.58,"pelvis":.50,"upper_thigh":.43}
meas={}
for k,zf in levels.items():
    meas[k]={"mona":slab(body,zf),"lap":slab(lapbody,zf)}

# Existing skeletal segment ratios from verified project measurements.
verified={
"upper_leg":{"lap":0.40910531516039506,"mona":0.3586978653628806},
"lower_leg":{"lap":0.43099702241739707,"mona":0.42772590987979087},
"foot_to_toe_base":{"lap":0.11746478611370792,"mona":0.1608908987813994},
"upper_arm":{"lap":0.19999641615894467,"mona":0.23925636913214893},
"lower_arm":{"lap":0.22955605844757296,"mona":0.26152933111074605},
"hip_joint_width":{"lap":0.13743670697774774,"mona":0.20820005238056188},
"upper_arm_head_width":{"lap":0.21998980582974342,"mona":0.22524438053382154},
"pelvis_to_shoulder_vertical":{"lap":0.41730235353393463,"mona":0.43883705139160156},
"pelvis_to_neck_vertical":{"lap":0.458064158563005,"mona":0.4826812744140625},
"pelvis_to_head_vertical":{"lap":0.5317365083525369,"mona":0.5628737211227417}
}
for v in verified.values():v["lap_over_mona"]=v["lap"]/v["mona"]

report={
"ok":True,"stage":"R_Master_v9_LapVsMona_BodyCompare",
"mona_source":mona,"lap_source":"Lapine_BaseBody.fbx",
"normalization":{"lap_height_scale_to_mona":s,"note":"comparison-only; no source assets modified"},
"mesh":{"mona_vertices":len(body.data.vertices),"lap_vertices":len(lapbody.data.vertices)},
"surface_slabs":meas,"verified_skeletal_measurements":verified,
"decision_scope":["body silhouette donor viability","head-neck cut planning","surface/material transfer planning"],
"source_modified":False,"rest_pose_baked":False,"weights_transferred":False,"final_vrm":False
}
with open(os.path.join(out,"R_Master_v9_report.json"),"w") as f:json.dump(report,f,indent=2)

# Render side-by-side overlay views.
scene=bpy.context.scene;scene.render.engine="BLENDER_EEVEE_NEXT";scene.render.resolution_x=760;scene.render.resolution_y=760;scene.render.resolution_percentage=100
scene.world.color=(.025,.025,.03);scene.view_settings.exposure=-.7
for o in list(bpy.data.objects):
    if o.type=="LIGHT":bpy.data.objects.remove(o,do_unlink=True)
def area(loc,en,size):
    d=bpy.data.lights.new("L","AREA");d.energy=en;d.size=size;o=bpy.data.objects.new("L",d);scene.collection.objects.link(o);o.location=Vector(loc);o.rotation_euler=(mcenter-o.location).to_track_quat("-Z","Y").to_euler()
area((mcenter.x-mh,mcenter.y-mh,mcenter.z+mh*.5),260,mh);area((mcenter.x+mh,mcenter.y-mh*.3,mcenter.z),100,mh)
cd=bpy.data.cameras.new("cam");cam=bpy.data.objects.new("cam",cd);scene.collection.objects.link(cam);scene.camera=cam;cam.data.type="ORTHO";cam.data.ortho_scale=mh*1.10
d=mh*2.8
views={
"front":(mcenter.x,mcenter.y-d,mcenter.z),
"side":(mcenter.x+d,mcenter.y,mcenter.z),
"back":(mcenter.x,mcenter.y+d,mcenter.z),
"three_quarter":(mcenter.x+d*.72,mcenter.y-d*.72,mcenter.z),
}
for name,pos in views.items():
    cam.location=Vector(pos);cam.rotation_euler=(mcenter-cam.location).to_track_quat("-Z","Y").to_euler();scene.render.filepath=os.path.join(out,f"R_Master_v9_{name}.png");bpy.ops.render.render(write_still=True)

# Region crops via camera target/scale, not destructive geometry.
regions={
"chest_waist":(mcenter+Vector((0,0,mh*.17)),mh*.42),
"waist_pelvis_thigh":(mcenter+Vector((0,0,-mh*.08)),mh*.46),
"head_neck_cut":(mcenter+Vector((0,0,mh*.38)),mh*.32),
}
for name,(tgt,sc) in regions.items():
    cam.data.ortho_scale=sc;cam.location=Vector((tgt.x,tgt.y-d,tgt.z));cam.rotation_euler=(tgt-cam.location).to_track_quat("-Z","Y").to_euler();scene.render.filepath=os.path.join(out,f"R_Master_v9_{name}.png");bpy.ops.render.render(write_still=True)

bpy.ops.wm.save_as_mainfile(filepath=os.path.join(out,"R_Master_v9_COMPARE_PREVIEW.blend"),check_existing=False)
print("V9_COMPARE_OK")
''',encoding="utf-8")

cmd=["xvfb-run","-a",str(BLENDER),"--background","--python",str(SCRIPT),"--","--mona",str(MONA),"--lap",str(FBX),"--out",str(OUT)]
r=subprocess.run(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
(OUT/"R_Master_v9_build.log").write_text(r.stdout,encoding="utf-8")
print(r.stdout[-8000:])
if r.returncode!=0: raise RuntimeError("v9 compare failed")

files=[
"R_Master_v9_front.png","R_Master_v9_side.png","R_Master_v9_back.png","R_Master_v9_three_quarter.png",
"R_Master_v9_chest_waist.png","R_Master_v9_waist_pelvis_thigh.png","R_Master_v9_head_neck_cut.png",
"R_Master_v9_report.json","R_Master_v9_build.log"
]
ZIP=OUT/"R_Master_v9_Review.zip"
if ZIP.exists():ZIP.unlink()
with zipfile.ZipFile(ZIP,"w",zipfile.ZIP_DEFLATED) as z:
    for f in files:
        p=OUT/f
        if p.exists():z.write(p,p.name)
print("✓ v9 Review ZIP:",ZIP,ZIP.stat().st_size,"bytes")

from IPython.display import display,Image,Markdown
for f in files[:7]:
    p=OUT/f
    if p.exists():
        display(Markdown("### "+f))
        display(Image(filename=str(p)))
from google.colab import files as cfiles
cfiles.download(str(ZIP))


